# Core Python Data Processing Practice

**Topic:** Python for Modeling &nbsp;·&nbsp; **Difficulty:** Easy

## The data (created in the next cell)

- `records`: roughly 730 dicts like `{"student": "stu042", "category": "hw", "score": 87.5}`.
- `weights`: `{"hw": 0.3, "quiz": 0.3, "exam": 0.4}`.
- Each student has many scores per category, but **some students are missing an entire category** (e.g., no exam scores).

## Your task

For each student compute a final grade:

1. Average their scores within each category they have.
2. Final grade = weighted average of those category means using `weights`, **renormalized over the categories the student actually has** (so a student with only hw and quiz uses weights 0.3/0.6 and 0.3/0.6).

Then fill in:

```python
answers = {
    "top_student": ...,  # str: student with the highest final grade
    "n_passing": ...,    # int: students with final grade >= 70
    "class_mean": ...,   # float: mean of all final grades
}
```

Use plain Python only in your solution (dicts/loops/comprehensions), no pandas or numpy.

## Evaluation

The evaluation cell recomputes the ground truth independently and checks all three answers.


In [ ]:
import numpy as np

rng = np.random.default_rng(11)

weights = {"hw": 0.3, "quiz": 0.3, "exam": 0.4}
students = [f"stu{i:03d}" for i in range(45)]
ability = {s: rng.uniform(55, 95) for s in students}
# a handful of students are missing one whole category
missing = {s: (rng.choice(["hw", "quiz", "exam"]) if rng.random() < 0.2 else None) for s in students}

records = []
for s in students:
    for cat, count in [("hw", rng.integers(8, 14)), ("quiz", rng.integers(4, 8)), ("exam", rng.integers(2, 4))]:
        if missing[s] == cat:
            continue
        for _ in range(int(count)):
            score = float(np.clip(ability[s] + rng.normal(0, 8), 0, 100))
            records.append({"student": s, "category": cat, "score": round(score, 1)})

rng.shuffle(records)
print(f"{len(records)} records, e.g. {records[0]}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your answers to the truth
# =====================================================================
if "answers" not in dir():
    raise NameError("Define the `answers` dict in the solution cell before running this cell.")

_sums = {}
for _r in records:
    _cell = _sums.setdefault(_r["student"], {}).setdefault(_r["category"], [0.0, 0])
    _cell[0] += _r["score"]; _cell[1] += 1
_finals = {}
for _s, _cats in _sums.items():
    _w = sum(weights[c] for c in _cats)
    _finals[_s] = sum(weights[c] * (t / n) for c, (t, n) in _cats.items()) / _w
_true_top = max(_finals, key=_finals.get)
_true_pass = sum(1 for v in _finals.values() if v >= 70)
_true_mean = sum(_finals.values()) / len(_finals)

_ok = 0
print("=" * 52)
_good = str(answers.get("top_student")) == _true_top; _ok += _good
print(f"  top_student yours={answers.get('top_student')!r:<12} {'CORRECT' if _good else 'WRONG'}")
_good = answers.get("n_passing") is not None and int(answers["n_passing"]) == _true_pass; _ok += _good
print(f"  n_passing   yours={answers.get('n_passing')!r:<12} {'CORRECT' if _good else 'WRONG'}")
_good = answers.get("class_mean") is not None and abs(float(answers["class_mean"]) - _true_mean) < 0.01; _ok += _good
print(f"  class_mean  yours={answers.get('class_mean')!r:<12} {'CORRECT' if _good else 'WRONG'}")
print("=" * 52)
print(f"Score: {_ok}/3")
print("PASSED: nice work!" if _ok == 3 else "NOT PASSED yet: check the renormalization rule.")


## Solution

Accumulate per-student, per-category sums and counts in nested dicts, average each category, then take the weighted mean renormalized over the categories the student actually has. The three answers fall out of one `finals` dict.

```python
sums = {}
for r in records:
    cell = sums.setdefault(r["student"], {}).setdefault(r["category"], [0.0, 0])
    cell[0] += r["score"]
    cell[1] += 1

finals = {}
for student, cats in sums.items():
    total_weight = sum(weights[c] for c in cats)
    finals[student] = sum(
        weights[c] * (total / count) for c, (total, count) in cats.items()
    ) / total_weight

answers = {
    "top_student": max(finals, key=finals.get),
    "n_passing": sum(1 for grade in finals.values() if grade >= 70),
    "class_mean": sum(finals.values()) / len(finals),
}
print(answers)
```
